<a href="https://colab.research.google.com/github/Lemonade-exe/python_research_2026/blob/PreCovidValues/STEP_ResearchQuestion_AlgorithmsV9(Scope0_100).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Meeting Notes
- 2/28/2026 Saturday
- Read and understand kNN algorithm
- run the with small sizes of parameters then in new colab notebooks extend the number of parameters gradually.



# Meeting Notes
- 2/21/2026 Saturday
- Read and understand kNN algorithm
- convert your code into nested for loops
- save your results in a dataframe




In [ ]:
# Libraries needed for data preparation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import yfinance as yf
import pandas as pd
import numpy as np
import csv
import requests

# Lag data / input
'''
def lag_func(data, name, lag):
    df_lag = pd.DataFrame(data[name])
    for i in range(1, lag+1):
        df_lag[f'lag_{i}'] = df_lag[name].shift(i)
        df_lag.dropna(inplace=True)
    return df_lag
'''

# Close data / output
def lag_func(data, name, lag):
    df_lag = data[['Date', name]].copy()

    for i in range(1, lag + 1):
        df_lag[f'lag_{i}'] = df_lag[name].shift(i)

    df_lag.dropna(inplace=True)
    return df_lag

# Data Preparation for Google
# Time Range
                # 1 month, 6 month, 1 year, 5 year, 10 year
startValues = ["2005-1-1", "2006-1-1", "2007-1-1", "2009-1-1", "2015-1-1"]
endValues = ["2005-2-1", "2006-7-1", "2008-1-1", "2014-1-1", "2025-1-1"]

START = "2010-1-1"
END = "2024-12-31"

# Creates/declare Dataframe
df = pd.DataFrame()

# Initialize the dataframe
df['Google'] = yf.Ticker('GOOG').history(start=START, end=END).Close

  # Removes hour
df.reset_index(inplace=True)
df['Date'] = [i.date() for i in df.Date]
df['Up/Down'] = (df['Google'].shift(-1) > df['Google']).astype(int)
# return dataframe
lag_func(df,'Google',5).head().round(4) ## Missing Date

,Date,Google,lag_1,lag_2,lag_3,lag_4,lag_5
5,2010-01-11,14.8497,14.8722,14.6765,15.0264,15.4149,15.4831
6,2010-01-12,14.5871,14.8497,14.8722,14.6765,15.0264,15.4149
7,2010-01-13,14.5034,14.5871,14.8497,14.8722,14.6765,15.0264
8,2010-01-14,14.5716,14.5034,14.5871,14.8497,14.8722,14.6765
9,2010-01-15,14.3282,14.5716,14.5034,14.5871,14.8497,14.8722


In [ ]:
#lagged df
df_lagged = lag_func(df, 'Google', 5)

#lagged data
df_lagged['Up/Down'] = df['Up/Down'].iloc[df_lagged.index]
df_lagged.dropna(inplace=True)
x, y = df_lagged[[f'lag_{i}' for i in range(1, 6)]], df_lagged['Up/Down']

# checks the amount of data
print(x.shape)
print(y.shape)

(3768, 5)
(3768,)


In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

#splitting data
split = int(len(x_scaled) * 0.8)

# declaring the x_train, y_train, x_test, y_test
x_train, x_test = x_scaled[:split], x_scaled[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# KNN

In [ ]:
# KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

# DataFrame declaration and initialization
KNN_df = pd.DataFrame(columns = ['N_neighbors', 'Weights', 'Algorithm', 'Leaf Size', 'Power', 'Train Score', 'Test Score'])

# Parameter Set
KNN_algorithm = {"auto", "ball_tree", "kd_tree", "brute"}
KNN_weights = {None, "uniform", "distance"}


# Model (n_neighbors)
for NNeighbors in range(20,120,20):
  for Weights in KNN_weights:
    for Algorithm in KNN_algorithm:
      for LeafSize in range(20,120,20):
        for Power in np.arange(20,120,20): # (42,45) (53,55) (55,57)
          knn = KNeighborsClassifier(n_neighbors = NNeighbors, weights = Weights, algorithm = Algorithm, leaf_size = LeafSize, p = Power)
          knn.fit(x_train, y_train)

          # Prediction
          knn.predict(x_train[:5]), knn.predict(x_test[:5])

          # Score
          trainScore = knn.score(x_train, y_train)
          testScore = knn.score(x_test, y_test)

          # Update DataFrame
          KNN_df.loc[len(KNN_df)] = [NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore]

          if(testScore >= 0.6):
            key = False
            print(NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore)



In [ ]:
KNN_df.round(2).sort_values('Test Score', ascending=False)

,N_neighbors,Weights,Algorithm,Leaf Size,Power,Train Score,Test Score
1499,100,uniform,brute,100,100,0.55,0.53
1493,100,uniform,brute,80,80,0.55,0.53
1494,100,uniform,brute,80,100,0.55,0.53
1495,100,uniform,brute,100,20,0.55,0.53
1496,100,uniform,brute,100,40,0.55,0.53
...,...,...,...,...,...,...,...
27,20,None,auto,20,60,0.61,0.46
32,20,None,auto,40,60,0.61,0.46
37,20,None,auto,60,60,0.61,0.46
7,20,None,ball_tree,40,60,0.61,0.46


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
KNN_df.to_csv('/content/drive/My Drive/KNNBigScope.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Decision Tree

In [ ]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

# DataFrame declaration and initialization
DT_df = pd.DataFrame(columns = ['Max Depth', 'Minimum Samples Split', 'Minimum Samples Leaf', 'Max Features', 'Random State', 'Maximum Leaf Nodes', 'Train Score', 'Test Score'])

# Model (max_depth)
for MaxDepth in range(20,120,20):
  for MinSamplesSplit in range(20,120,20):
    for MinSamplesLeaf in range(20,120,20):
      for MaxFeatures in range(20,120,20):
        for RandomState in range(20,120,20):
          for MaxLeafNodes in range(20,120,20):
            # Model
            dt = DecisionTreeClassifier(max_depth = MaxDepth, min_samples_split = MinSamplesSplit, min_samples_leaf = MinSamplesLeaf, max_features = MaxFeatures, random_state = RandomState, max_leaf_nodes = MaxLeafNodes)
            dt.fit(x_train, y_train)

            # Prediction
            (dt.predict(x_train[:5]) >= 0.5).astype(int)
            (dt.predict(x_test[:5]) >= 0.5).astype(int)

            # Score
            trainScore = dt.score(x_train,y_train)
            testScore = dt.score(x_test,y_test)

            # Update DataFrame
            DT_df.loc[len(DT_df)] = [MaxDepth, MinSamplesSplit, MinSamplesLeaf, MaxFeatures, RandomState, MaxLeafNodes, trainScore, testScore]


In [ ]:
DT_df.round(2).sort_values('Test Score', ascending = False)

,Max Depth,Minimum Samples Split,Minimum Samples Leaf,Max Features,Random State,Maximum Leaf Nodes,Train Score,Test Score
15608,100.0,100.0,100.0,100.0,40.0,80.0,0.59,0.54
15607,100.0,100.0,100.0,100.0,40.0,60.0,0.59,0.54
9256,60.0,100.0,100.0,20.0,40.0,40.0,0.59,0.54
9255,60.0,100.0,100.0,20.0,40.0,20.0,0.58,0.54
9254,60.0,100.0,100.0,20.0,20.0,100.0,0.59,0.54
...,...,...,...,...,...,...,...,...
9125,60.0,100.0,80.0,20.0,20.0,20.0,0.58,0.48
9126,60.0,100.0,80.0,20.0,20.0,40.0,0.60,0.48
9127,60.0,100.0,80.0,20.0,20.0,60.0,0.60,0.48
9128,60.0,100.0,80.0,20.0,20.0,80.0,0.60,0.48


In [ ]:
DT_df.to_csv('/content/drive/My Drive/DecisionTreeBigScope.csv', index=False)

# Random Forest

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

# DataFrame declaration and initialization
RF_df = pd.DataFrame(columns = ['N_Estimators', 'Random State', 'Minimum Samples Split', 'Verbose', 'n_jobs', 'Train Score', 'Test Score'])

for NEstimators in range (1,91,30):
  for RandomState in range(0,91,30):
    for MinSamplesSplit in range(2,92,30):
      for ver in range(0,90,30):
        for nJobs in range (1,91,30):
          # Model
          rf = RandomForestClassifier(n_estimators = NEstimators, random_state = RandomState, min_samples_split = MinSamplesSplit, verbose = ver, n_jobs = nJobs)
          rf.fit(x_train,y_train)

          # Prediction
          (rf.predict(x_train[:5]) >= 0.5).astype(int)
          (rf.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = rf.score(x_train,y_train)
          testScore = rf.score(x_test,y_test)

          # Update DataFrame
          RF_df.loc[len(RF_df)] = [NEstimators, RandomState, MinSamplesSplit, ver, nJobs, trainScore, testScore]



building tree 1 of 1


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=31)]: Using backend ThreadingBackend with 31 concurrent workers.


building tree 1 of 1


[Parallel(n_jobs=31)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=61)]: Using backend ThreadingBackend with 61 concurrent workers.
[Parallel(n_jobs=61)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Paralle

building tree 1 of 1
building tree 1 of 1
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=31)]: Using backend ThreadingBackend with 31 concurrent workers.
building tree 1 of 1
[Parallel(n_jobs=31)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parall

...

building tree 1 of 61building tree 2 of 61
building tree 3 of 61
building tree 4 of 61
building tree 5 of 61
building tree 6 of 61
building tree 7 of 61
building tree 8 of 61

building tree 9 of 61
building tree 10 of 61
building tree 11 of 61
building tree 12 of 61
building tree 13 of 61
building tree 14 of 61
building tree 15 of 61
building tree 16 of 61
building tree 17 of 61
building tree 18 of 61
building tree 19 of 61
building tree 20 of 61
building tree 21 of 61
building tree 22 of 61
building tree 23 of 61
building tree 24 of 61
building tree 25 of 61
building tree 26 of 61
building tree 27 of 61
building tree 28 of 61
building tree 29 of 61


[Parallel(n_jobs=61)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=61)]: Done   3 out of  61 | elapsed:    0.3s remaining:    5.8s


building tree 30 of 61
building tree 31 of 61
building tree 32 of 61
building tree 33 of 61
building tree 34 of 61
building tree 35 of 61
building tree 36 of 61
building tree 37 of 61
building tree 38 of 61
building tree 39 of 61
building tree 40 of 61
building tree 41 of 61
building tree 42 of 61
building tree 43 of 61
building tree 44 of 61
building tree 45 of 61
building tree 46 of 61
building tree 47 of 61
building tree 48 of 61
building tree 49 of 61
building tree 50 of 61
building tree 51 of 61
building tree 52 of 61
building tree 53 of 61
building tree 54 of 61
building tree 55 of 61
building tree 56 of 61
building tree 57 of 61
building tree 58 of 61
building tree 59 of 61
building tree 60 of 61
building tree 61 of 61


[Parallel(n_jobs=61)]: Done   6 out of  61 | elapsed:    0.5s remaining:    4.8s
[Parallel(n_jobs=61)]: Done   9 out of  61 | elapsed:    0.6s remaining:    3.5s
[Parallel(n_jobs=61)]: Done  12 out of  61 | elapsed:    0.6s remaining:    2.5s
[Parallel(n_jobs=61)]: Done  15 out of  61 | elapsed:    0.6s remaining:    1.9s
[Parallel(n_jobs=61)]: Done  18 out of  61 | elapsed:    0.6s remaining:    1.4s
[Parallel(n_jobs=61)]: Done  21 out of  61 | elapsed:    0.6s remaining:    1.2s
[Parallel(n_jobs=61)]: Done  24 out of  61 | elapsed:    0.6s remaining:    0.9s
[Parallel(n_jobs=61)]: Done  27 out of  61 | elapsed:    0.6s remaining:    0.8s
[Parallel(n_jobs=61)]: Done  30 out of  61 | elapsed:    0.6s remaining:    0.6s
[Parallel(n_jobs=61)]: Done  33 out of  61 | elapsed:    0.7s remaining:    0.6s
[Parallel(n_jobs=61)]: Done  36 out of  61 | elapsed:    0.7s remaining:    0.5s
[Parallel(n_jobs=61)]: Done  39 out of  61 | elapsed:    0.8s remaining:    0.4s
[Parallel(n_jobs=61)]: Done 

building tree 1 of 61
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
building tree 2 of 61
[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:    0.0s
building tree 3 of 61
[Parallel(n_jobs=1)]: Done   3 tasks      | elapsed:    0.0s
building tree 4 of 61
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    0.1s
building tree 5 of 61
[Parallel(n_jobs=1)]: Done   5 tasks      | elapsed:    0.1s
building tree 6 of 61
[Parallel(n_jobs=1)]: Done   6 tasks      | elapsed:    0.1s
building tree 7 of 61
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    0.1s
building tree 8 of 61
[Parallel(n_jobs=1)]: Done   8 tasks      | elapsed:    0.1s
building tree 9 of 61
[Parallel(n_jobs=1)]: Done   9 tasks      | elapsed:    0.1s
building tree 10 of 61
[Parallel(n_jobs=1)]: Done  10 tasks      | elapsed:    0.1s
building tree 11 of 61
[Parallel(n_jobs=1)]: Done  11 tasks      | elapsed:    0.2s
building tree 12 of 61
[Parallel(n_jobs=1)]: Done  12 tasks      | elapsed:    0.2s
b

In [ ]:
RF_df.round(2).sort_values('Test Score', ascending = False)

,N_Estimators,Random State,Minimum Samples Split,Verbose,n_jobs,Train Score,Test Score
28,1.0,30.0,2.0,0.0,31.0,0.82,0.52
31,1.0,30.0,2.0,30.0,31.0,0.82,0.52
29,1.0,30.0,2.0,0.0,61.0,0.82,0.52
30,1.0,30.0,2.0,30.0,1.0,0.82,0.52
34,1.0,30.0,2.0,60.0,31.0,0.82,0.52
...,...,...,...,...,...,...,...
37,1.0,30.0,32.0,0.0,31.0,0.65,0.45
41,1.0,30.0,32.0,30.0,61.0,0.65,0.45
42,1.0,30.0,32.0,60.0,1.0,0.65,0.45
43,1.0,30.0,32.0,60.0,31.0,0.65,0.45


In [ ]:
RF_df.to_csv('/content/drive/My Drive/RFBigScope.csv', index=False)

# XGBoost

In [ ]:
# XGBoost Classifier
from xgboost import XGBClassifier

# DataFrame declaration and initialization
XGB_df = pd.DataFrame(columns = ['N_Estimators', 'Max Depth', 'Learning Rate', 'Max Leaves', 'n_jobs', 'Train Score', 'Test Score'])

for NEstimators in range(20,120,20):
  for MaxDepth in range(20,120,20):
    for LearningRate in range(20,120,20):
      for leaves in range (20,120,20):
        for nJobs in range (20,120,20):
          # Model
          xgb = XGBClassifier(n_estimators = NEstimators, max_depth = MaxDepth, learning_rate = LearningRate, max_leaves = leaves, n_jobs = nJobs)
          xgb.fit(x_train, y_train)

          # Prediction
          (xgb.predict(x_train[:5]) >= 0.5).astype(int)
          (xgb.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = xgb.score(x_train,y_train)
          testScore = xgb.score(x_test,y_test)

          # Update Dataframe
          XGB_df.loc[len(XGB_df)] = [NEstimators, MaxDepth, LearningRate, leaves, nJobs, trainScore, testScore]

In [ ]:
XGB_df.round(2).sort_values('Test Score', ascending = False)

,N_Estimators,Max Depth,Learning Rate,Max Leaves,n_jobs,Train Score,Test Score
5,20.0,20.0,20.0,40.0,20.0,0.49,0.53
3124,100.0,100.0,100.0,100.0,100.0,0.50,0.53
0,20.0,20.0,20.0,20.0,20.0,0.49,0.53
1,20.0,20.0,20.0,20.0,40.0,0.49,0.53
2,20.0,20.0,20.0,20.0,60.0,0.49,0.53
...,...,...,...,...,...,...,...
80,20.0,20.0,80.0,40.0,20.0,0.50,0.45
81,20.0,20.0,80.0,40.0,40.0,0.50,0.45
82,20.0,20.0,80.0,40.0,60.0,0.50,0.45
83,20.0,20.0,80.0,40.0,80.0,0.50,0.45


In [ ]:
XGB_df.to_csv('/content/drive/My Drive/XGBBigScope.csv', index=False)

# Logistic Regression

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

# DataFrame Declaration and initialization
Log_df = pd.DataFrame(columns = ['Penalty', 'Inverse of Regularization Strength', 'Elastic-Net Mixing','Max Iteration', 'Verbose','Train Score', 'Test Score'])

# Parameter Set
Log_penalty = {"l1", "l2", "elasticnet", None}

# Model
for Penalty in Log_penalty:
  for InverseRegStren in np.arange(20,120,20):
    for MaxIter in range(200,1200,200):
      for Verbose in range(0,3):
        if(Penalty == 'elasticnet'):
          for ElasticNetMix in np.arange(0,1):
            # Model
            log = LogisticRegression(penalty = Penalty, C = InverseRegStren,l1_ratio = ElasticNetMix, solver = "saga", max_iter = MaxIter, verbose = Verbose)
            log.fit(x_train, y_train)

            # Prediction
            (log.predict(x_train[:5]) >= 0.5).astype(int)
            (log.predict(x_test[:5]) >= 0.5).astype(int)

            # Score
            trainScore = log.score(x_train, y_train)
            testScore = log.score(x_test, y_test)

            # Update Dataframe
            Log_df.loc[len(Log_df)] = [Penalty, InverseRegStren, ElasticNetMix, MaxIter, Verbose, trainScore, testScore]
        elif(Penalty == 'l1'):
          # Model
          log = LogisticRegression(penalty = Penalty, C = InverseRegStren,l1_ratio = None, solver = "saga", max_iter = MaxIter, verbose = Verbose)
          log.fit(x_train, y_train)

          # Prediction
          (log.predict(x_train[:5]) >= 0.5).astype(int)
          (log.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = log.score(x_train, y_train)
          testScore = log.score(x_test, y_test)

          # Update Dataframe
          Log_df.loc[len(Log_df)] = [Penalty, InverseRegStren, None, MaxIter, Verbose, trainScore, testScore]
        else:
          # Model
          log = LogisticRegression(penalty = Penalty, C = InverseRegStren,l1_ratio = None, max_iter = MaxIter, verbose = Verbose)
          log.fit(x_train, y_train)

          # Prediction
          (log.predict(x_train[:5]) >= 0.5).astype(int)
          (log.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = log.score(x_train, y_train)
          testScore = log.score(x_test, y_test)

          # Update Dataframe
          Log_df.loc[len(Log_df)] = [Penalty, InverseRegStren, None, MaxIter, Verbose, trainScore, testScore]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.w

max_iter reached after 0 seconds
max_iter reached after 1 seconds


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


max_iter reached after 0 seconds


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s finished


...

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.3s finished


convergence after 462 epochs took 1 seconds


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s finished


convergence after 460 epochs took 0 seconds


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s finished


In [ ]:
Log_df.round(2).sort_values('Test Score', ascending=False)

,Penalty,Inverse of Regularization Strength,Elastic-Net Mixing,Max Iteration,Verbose,Train Score,Test Score
271,elasticnet,80.0,0,200.0,1.0,0.53,0.55
270,elasticnet,80.0,0,200.0,0.0,0.53,0.55
257,elasticnet,60.0,0,200.0,2.0,0.53,0.55
256,elasticnet,60.0,0,200.0,1.0,0.53,0.55
255,elasticnet,60.0,0,200.0,0.0,0.53,0.55
...,...,...,...,...,...,...,...
295,elasticnet,100.0,0,800.0,1.0,0.53,0.54
296,elasticnet,100.0,0,800.0,2.0,0.53,0.54
297,elasticnet,100.0,0,1000.0,0.0,0.53,0.54
298,elasticnet,100.0,0,1000.0,1.0,0.53,0.54


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
Log_df.to_csv('/content/drive/My Drive/LogScope(0-100).csv', index=False)

Mounted at /content/drive


l1_ratio and solver must be saga when it is elasticnet
else solver must be liblinear for everything else
